# Burst Analysis Quickstart

This notebook runs the same burst-analysis pipeline on either the `stimRemovalNull` NPZ recordings or the raw H5 recordings. Select the dataset in the configuration cell; downstream analysis cells use the common `RestingActivityDataset`/`Recording` objects and should not need dataset-specific edits.

For now it:

- loads one configured recording for detailed inspection,
- selects reference electrodes using the usual `PrepConfig`,
- shows the initial layout-grid and pooled IFR views via the standard `ephax` analyzers,
- computes per-electrode instantaneous firing-rate (IFR) traces on a shared time grid,
- separates low-activity, high-activity, and burst periods,
- builds aggregate activity-state and wave-speed summaries across configured wells/DIVs where available.


In [ ]:
%matplotlib inline

import os
import sys
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import imageio.v2 as imageio

repo_root = Path.cwd()
if not (repo_root / "ephax").exists() and (repo_root.parent / "ephax").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

os.environ.setdefault("MPLCONFIGDIR", str((repo_root / ".mpl-cache").resolve()))

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.ticker import LogFormatterMathtext, LogLocator
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks, peak_prominences
from IPython.display import Image, display
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from ephax import RestingActivityDataset, PrepConfig, LayoutGridPlotter
from ephax.analyzers import IFRAnalyzer
from ephax.analyzers.ifr import IFRConfig
from ephax.helper_functions import calculate_ifr
from ephax.models import WaveAnalysisResult
from ephax.plotting import (
    plot_high_activity_burst_windows,
    plot_macro_burst_detector_comparison_windows,
    plot_activity_state_ifr_kde_histograms,
)
from ephax.metrics import (
    activity_state_kde_peak_frequencies,
    align_highres_to_anchors,
    binned_kde_peak_summary,
    analyze_eventwise_waves,
    fit_wave_speed,
    build_highres_traces,
    build_network_activity_state,
    build_participation_activity_state,
    build_population_ifr,
    build_trigger_summary,
    detect_coarse_burst_epochs,
    detect_nested_gamma_anchors,
    detect_network_burst_epochs,
    detect_participation_burst_epochs,
    detect_high_activity_epochs,
    extract_activity_state_ifr,
    summarize_wave_peaks,
)

plt.rcParams["figure.dpi"] = 120
np.random.seed(0)


## Configure Recording and Burst View

The defaults below mirror the existing quickstart style: a 300 s recording-relative window and top-active electrode selection.


In [ ]:
# Dataset switch: "stim_removal_null" or "raw_h5".
DATASET = "stim_removal_null"

# Single-recording selection. DIV is used by stimRemovalNull and ignored for raw H5.
WELL = 0
DIV = 21
RAW_H5_DIV = 40

STIM_DATA_ROOT = repo_root / "ephax/data/stimRemovalNull"
STIM_DATA_FILENAME_TEMPLATE = "DIV{div}_240703_data_well{well}_exp_data.npz"

H5_FILE_INFO = {
    0: ("ephax/data", "well_0.raw.h5", 0),
    1: ("ephax/data", "well_1.raw.h5", 1),
    2: ("ephax/data", "well_2_3.raw.h5", 2),
    3: ("ephax/data", "well_2_3.raw.h5", 3),
    4: ("ephax/data", "well_4.raw.h5", 4),
    5: ("ephax/data", "well_5.raw.h5", 5),
}

AGGREGATE_WELLS = [0, 1, 2, 3, 4, 5]
AGGREGATE_DIVS = [DIV]

START_SEC = 0.0
END_SEC = 300.0 if DATASET == "stim_removal_null" else 600.0
MIN_AMP = 0.0

TOP_START = 0
TOP_STOP = 1000 if DATASET == "stim_removal_null" else 300

IFR_GRID_HZ = 50.0
IFR_MAX_HZ = 1000.0
SMOOTH_SIGMA_SEC = 0.15

BURST_MIN_DISTANCE_SEC = 1.0
BURST_PROMINENCE_QUANTILE = 0.90
BURST_PROMINENCE_SCALE = 0.20
BURST_WINDOW_SEC = 2.5

COARSE_EPOCH_REL_HEIGHT = 0.20
HIGHRES_BIN_MS = 1.0
HIGHRES_SMOOTH_SIGMA_MS = 3.0
GAMMA_SEARCH_MS = 120.0
GAMMA_SEARCH_TO_EPOCH_END = True
GAMMA_MIN_DISTANCE_MS = 40.0
GAMMA_PROMINENCE_FRAC = 0.08
GAMMA_PROMINENCE_ABS_FLOOR = 0.10
GAMMA_KEEP_HEIGHT_FRAC = 0.50

NETWORK_BIN_MS = 10.0
NETWORK_ACTIVE_RATE_FLOOR_HZ = 1.0
NETWORK_THRESHOLD_BASELINE_QUANTILE = 0.20
NETWORK_THRESHOLD_IQR_SCALE = 3.0
NETWORK_MIN_PARTICIPATION_FRACTION = 0.05
NETWORK_MIN_ACTIVE_ELECTRODES = 10
NETWORK_MERGE_GAP_MS = 50.0
NETWORK_MIN_DURATION_MS = 20.0
NETWORK_MIN_SPIKES = 20
HIGH_ACTIVITY_MAD_SCALE = 3.0
HIGH_ACTIVITY_MIN_DURATION_MS = 30.0
HIGH_ACTIVITY_MAX_GAP_BINS = 0
BURST_ANCHOR_WINDOW_MS = NETWORK_BIN_MS

ALIGN_PRE_MS = 20.0
ALIGN_POST_MS = 40.0
COND_PROP_DELAY_START_MS = -10.0
COND_PROP_DELAY_STOP_MS = 30.0
COND_PROP_WINDOW_HALF_WIDTH_MS = 1.0
COND_PROP_DISTANCE_BIN_UM = 100.0
COND_PROP_TRIGGER_SCOPE = "gamma"  # "gamma", "macro_burst", or "non_burst"
WAVE_X_BIN_UM = 300.0
WAVE_PEAK_SEARCH_START_MS = -15.0
WAVE_PEAK_SEARCH_STOP_MS = 20.0
WAVE_TRACE_SMOOTH_SIGMA_MS = 2.0
WAVE_MIN_ELECTRODES_PER_BIN = 5
WAVE_MIN_EVENTS_PER_BIN = 5
WAVE_BOOTSTRAP_REPS = 2000
WAVE_RANDOM_SEED = 0
WAVE_CACHE_ROOT = repo_root / "outputs" / f"{DATASET}_wave_results"
WAVE_FORCE_RECOMPUTE = False
AGGREGATE_WAVE_FORCE_RECOMPUTE = False
SAVE_INLINE_PLOTS = False
INLINE_PLOT_DPI = 180

SORT_PEAK_PRE_MS = 5.0
SORT_PEAK_POST_MS = 10.0
GIF_FRAME_STEP_MS = 2.0
EXAMPLE_GAMMA_EVENT_COUNT = 6
MACRO_BURST_ZOOM_COUNT = 4
MACRO_BURST_ZOOM_PAD_SEC = 0.40
EXAMPLE_EVENT_HEX_SAVE = False
EXAMPLE_EVENT_HEX_DIR = Path("GIFs/example_event_hex")
HEX_GRID_SIZE = 35 if DATASET == "stim_removal_null" else 50
ARRAY_X_MIN_UM = 0.0
ARRAY_Y_MIN_UM = 0.0
ARRAY_X_MAX_UM = 3850.0
ARRAY_Y_MAX_UM = 2100.0


def stim_recording_path(well, div):
    return STIM_DATA_ROOT / f"well{int(well)}" / STIM_DATA_FILENAME_TEMPLATE.format(div=int(div), well=int(well))


def build_recording_spec(well, div=None):
    if DATASET == "stim_removal_null":
        div = DIV if div is None else div
        path = stim_recording_path(well, div)
        if not path.exists():
            raise FileNotFoundError(f"Could not find stimRemovalNull recording file: {path}")
        return {
            "dataset": DATASET,
            "source": "npz",
            "path": path,
            "well": int(well),
            "div": int(div),
            "recording_id": f"stimRemovalNull_well{int(well)}_DIV{int(div)}",
            "label": f"stimRemovalNull well{int(well)} DIV{int(div)}",
            "source_file": path.name,
        }

    if DATASET == "raw_h5":
        if int(well) not in H5_FILE_INFO:
            raise ValueError(f"Unsupported raw H5 WELL={well}; available wells: {sorted(H5_FILE_INFO)}")
        h5_folder, h5_filename, h5_well_index = H5_FILE_INFO[int(well)]
        folder_path = repo_root / h5_folder
        file_path = folder_path / h5_filename
        if not file_path.exists():
            raise FileNotFoundError(f"Could not find raw H5 recording file: {file_path}")
        return {
            "dataset": DATASET,
            "source": "h5",
            "folder": folder_path,
            "filename": h5_filename,
            "path": file_path,
            "well": int(well),
            "h5_well_index": int(h5_well_index),
            "div": int(RAW_H5_DIV),
            "recording_id": f"raw_h5_well{int(well)}_DIV{int(RAW_H5_DIV)}",
            "label": f"raw H5 well{int(well)} DIV{int(RAW_H5_DIV)}",
            "source_file": h5_filename,
        }

    raise ValueError("DATASET must be 'stim_removal_null' or 'raw_h5'.")


def iter_aggregate_recording_specs():
    if DATASET == "stim_removal_null":
        for div in AGGREGATE_DIVS:
            for well in AGGREGATE_WELLS:
                path = stim_recording_path(well, div)
                if not path.exists():
                    print(f"Skipping missing recording: well{well} DIV{div}")
                    continue
                yield build_recording_spec(well, div)
        return

    for well in AGGREGATE_WELLS:
        if int(well) not in H5_FILE_INFO:
            print(f"Skipping unsupported raw H5 well: {well}")
            continue
        yield build_recording_spec(well)


def load_recording_from_spec(spec, *, start_sec=START_SEC, end_sec=END_SEC, min_amp=MIN_AMP):
    if spec["source"] == "npz":
        with np.load(spec["path"], allow_pickle=True) as data:
            sf_native = float(np.asarray(data["samp_rate"]).reshape(-1)[0])
            frameno = np.asarray(data["spike_data"]["frameno"], dtype=float)
            rec_t0 = float(np.min(frameno) / sf_native) if frameno.size else 0.0
        file_info = [(str(spec["path"]), rec_t0 + float(start_sec), rec_t0 + float(end_sec), int(spec["well"]))]
        return RestingActivityDataset.from_file_info(file_info, source="npz", min_amp=float(min_amp))

    if spec["source"] == "h5":
        file_info = [
            (str(spec["folder"]), spec["filename"], float(start_sec), float(end_sec), int(spec["h5_well_index"]))
        ]
        return RestingActivityDataset.from_file_info(file_info, source="h5", min_amp=float(min_amp))

    raise ValueError(f"Unsupported recording source: {spec['source']}")


recording_spec = build_recording_spec(WELL, DIV)
recording_label = recording_spec["label"]
GIF_OUTPUT_PATH = Path(f"GIFs/{recording_spec['recording_id']}_gamma_ifr_grid.gif")

recording_spec

INLINE_PLOT_OUTPUT_DIR = repo_root / "outputs" / "burst_analysis_inline_plots" / recording_spec["recording_id"]


def plot_filename(label):
    text = str(label).strip().lower()
    keep = []
    previous_underscore = False
    for char in text:
        if char.isalnum():
            keep.append(char)
            previous_underscore = False
        else:
            if not previous_underscore:
                keep.append("_")
                previous_underscore = True
    return "".join(keep).strip("_") or "plot"


def save_inline_plot(fig, name, *, show=True):
    if SAVE_INLINE_PLOTS:
        INLINE_PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        path = INLINE_PLOT_OUTPUT_DIR / f"{plot_filename(name)}.png"
        fig.savefig(path, dpi=int(INLINE_PLOT_DPI), bbox_inches="tight")
        print(f"Saved plot: {path}")
    if show:
        plt.show()
    return fig


In [ ]:
ds = load_recording_from_spec(recording_spec)
rec = ds.recordings[0]

in_window = (rec.spikes["time"] >= rec.start_time) & (rec.spikes["time"] <= rec.end_time)
duration = float(rec.end_time - rec.start_time)
n_spikes = int(np.sum(in_window))
n_active_electrodes = int(np.unique(rec.spikes["electrode"][in_window]).size)

print(f"Loaded 1 recording: {recording_spec['label']}")
print(f"Source: {recording_spec['source_file']}")
print(f"Window: [{rec.start_time:.4f}, {rec.end_time:.4f}] s ({duration:.1f} s)")
print(f"Sampling rate: {rec.sf:.1f} Hz | spikes in window: {n_spikes:,} | active electrodes in window: {n_active_electrodes}")


## Select Reference Electrodes

Use the same `PrepConfig` pattern as elsewhere in the repo, then keep the selected electrodes fixed for the burst analysis steps below.


In [ ]:
prep_cfg = PrepConfig(mode="top", top_start=TOP_START, top_stop=TOP_STOP, verbose=False)
refs = ds.select_ref_electrodes(prep_cfg)[0]

print(f"Selected {refs.size} electrodes")
print("First 20 selected electrode ids:", refs[:20])


## Initial Layout and IFRAnalyzer Views

Before the burst-specific plots below, reuse the standard `ephax` visualizers on the same single-recording dataset and `PrepConfig` selection.


In [ ]:
recording_labels = [recording_label]

lg = LayoutGridPlotter(ds)
fig_layout, _ = lg.plot_grid_avghz_panel(
    grid_size=50.0,
    ncols=1,
    interpolate=False,
    title=f"{recording_label}: Layout Grid",
    recording_titles=recording_labels,
)
save_inline_plot(fig_layout, "layout_grid_average_hz", show=False)
fig_layout_interp, _ = lg.plot_grid_avghz_panel(
    grid_size=50.0,
    ncols=1,
    interpolate=True,
    title=f"{recording_label}: Layout Grid (interpolated)",
    recording_titles=recording_labels,
)
save_inline_plot(fig_layout_interp, "layout_grid_average_hz_interpolated", show=False)

ifr_hist_bins = 200
ifr_cfg = IFRConfig(
    log_scale=True,
    overlay_gmm=True,
    ts_bins=200,
    time_grid_hz=200.0,
    max_time_points=2_000,
)
ifr_analyzer = IFRAnalyzer.from_dataset(ds, config=ifr_cfg, selection_prep_config=prep_cfg)
peaks = ifr_analyzer.peaks()

print(f"Collected {peaks.values.size} IFR samples")
if peaks.peaks_hz.size:
    print("GMM peak locations (Hz):", np.round(peaks.peaks_hz, 2))
else:
    print("No IFR peaks detected; consider widening the selection window.")

fig_ifr_hist, _ = ifr_analyzer.plot_histogram(hist_bins=ifr_hist_bins, show=True)
save_inline_plot(fig_ifr_hist, "ifr_histogram_gmm", show=False)
for panel_idx, (fig_ts, _axes_ts) in enumerate(
    ifr_analyzer.plot_timeseries(
        recording_titles=recording_labels,
        title=f"{recording_label}: IFRAnalyzer",
    )
):
    save_inline_plot(fig_ts, f"ifr_timeseries_panel_{panel_idx}", show=False)


## KDE-Based IFR Histograms

Plot pooled IFR samples on both linear and logarithmic axes using the same Gaussian KDE estimator. The maxima are extracted from the smoothed KDE curves in each coordinate system so we can compare dominant IFR modes without relying only on histogram binning.



In [ ]:
def collect_pooled_ifr_values_hz(dataset, selected_refs_per_recording):
    spikes_list, _layouts, start_times, end_times = dataset.to_legacy()
    pooled = []
    for spikes_data, start_time, end_time, selected_refs in zip(
        spikes_list,
        start_times,
        end_times,
        selected_refs_per_recording,
    ):
        _, _, ifr_vals = calculate_ifr(spikes_data, selected_refs, start_time, end_time)
        pooled.extend(ifr_vals)
    pooled = np.asarray(pooled, dtype=float)
    pooled = pooled[np.isfinite(pooled) & (pooled > 0) & (pooled <= IFR_MAX_HZ)]
    return pooled


ifr_positive_hz = collect_pooled_ifr_values_hz(ds, ifr_analyzer._refs_per_recording)
linear_hist = binned_kde_peak_summary(
    ifr_positive_hz,
    log_bins=False,
    n_bins=180,
    grid_size=4096,
    prominence_fraction=0.02,
    distance_fraction=0.01,
    bandwidth_scale=0.35,
    max_hz=IFR_MAX_HZ,
)
log_hist = binned_kde_peak_summary(
    ifr_positive_hz,
    log_bins=True,
    n_bins=180,
    grid_size=4096,
    prominence_fraction=0.02,
    distance_fraction=0.01,
    bandwidth_scale=0.35,
    max_hz=IFR_MAX_HZ,
)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), constrained_layout=True)
ax0, ax1 = axes

linear_widths = np.diff(linear_hist['plot_edges_hz'])
ax0.bar(
    linear_hist['plot_centers_hz'],
    linear_hist['counts'],
    width=linear_widths,
    color='0.82',
    edgecolor='0.55',
    align='center',
)
ax0.plot(linear_hist['grid_hz'], linear_hist['smoothed_counts'], color='black', lw=2.0)
ax0.scatter(linear_hist['peak_hz'], linear_hist['peak_counts'], color='crimson', s=32, zorder=3)
for peak_hz, peak_counts in zip(linear_hist['peak_hz'][:6], linear_hist['peak_counts'][:6]):
    ax0.text(float(peak_hz), float(peak_counts), f"{peak_hz:.2f}", fontsize=8, ha='left', va='bottom')
ax0.set_xlabel('Instantaneous firing rate (Hz)')
ax0.set_ylabel('Count')
ax0.set_title('Linear-bin IFR histogram with binned-KDE maxima')

log_widths = np.diff(log_hist['plot_edges_hz'])
ax1.bar(
    log_hist['plot_centers_hz'],
    log_hist['counts'],
    width=log_widths,
    color='0.82',
    edgecolor='0.55',
    align='center',
)
ax1.plot(log_hist['grid_hz'], log_hist['smoothed_counts'], color='black', lw=2.0)
ax1.scatter(log_hist['peak_hz'], log_hist['peak_counts'], color='crimson', s=32, zorder=3)
for peak_hz, peak_counts in zip(log_hist['peak_hz'][:6], log_hist['peak_counts'][:6]):
    ax1.text(float(peak_hz), float(peak_counts), f"{peak_hz:.2f}", fontsize=8, ha='left', va='bottom')
ax1.set_xscale('log')
ax1.set_xlabel('Instantaneous firing rate (Hz)')
ax1.set_ylabel('Count')
ax1.set_title('Log-bin IFR histogram with binned-KDE maxima')

save_inline_plot(fig, "pooled_ifr_binned_kde_maxima")

ifr_kde_peak_summary = pd.DataFrame(
    {
        'axis': ['linear'] * linear_hist['peak_hz'].size + ['log_bins'] * log_hist['peak_hz'].size,
        'peak_hz': np.concatenate([linear_hist['peak_hz'], log_hist['peak_hz']]),
        'peak_count': np.concatenate([linear_hist['peak_counts'], log_hist['peak_counts']]),
    }
).sort_values(['axis', 'peak_count'], ascending=[True, False], ignore_index=True)

print('Top binned-KDE peaks (linear bins):', np.round(linear_hist['peak_hz'][:6], 3))
print('Top binned-KDE peaks (log bins):', np.round(log_hist['peak_hz'][:6], 3))
ifr_kde_peak_summary.head(12)


## Build a Shared IFR Matrix

Each selected electrode gets interpolated onto the same time grid. The burst population trace is the pointwise mean across the selected electrodes.


In [ ]:
population = build_population_ifr(
    rec,
    refs,
    grid_hz=IFR_GRID_HZ,
    smooth_sigma_sec=SMOOTH_SIGMA_SEC,
)
burst_data = population

time_grid = population.time_grid
ifr_matrix = population.ifr_matrix
mean_ifr = population.mean_ifr
mean_ifr_smooth = population.mean_ifr_smooth

def plot_ifr_summary(time_axis, ifr_view, mean_view, mean_smooth_view, heatmap_title, mean_title, mean_log_scale=False, plot_name=None):
    positive_ifr = ifr_view[ifr_view > 0]
    if positive_ifr.size == 0:
        raise ValueError("IFR view contains no positive values for log-scale plotting.")

    heatmap_vmin = max(1e-3, float(np.quantile(positive_ifr, 0.01)))
    heatmap_vmax = float(positive_ifr.max())
    heatmap_vmax = max(heatmap_vmax, heatmap_vmin * (1.0 + 1e-6))
    mean_log_floor = max(heatmap_vmin, 1e-3)

    fig = plt.figure(figsize=(14, 8), constrained_layout=True)
    gs = fig.add_gridspec(
        2,
        2,
        height_ratios=[3.0, 1.6],
        width_ratios=[40.0, 1.6],
    )
    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[1, 0], sharex=ax0)
    cax = fig.add_subplot(gs[0, 1])
    fig.add_subplot(gs[1, 1]).axis("off")

    display_ifr = np.clip(ifr_view, heatmap_vmin, heatmap_vmax)
    im = ax0.imshow(
        display_ifr,
        aspect="auto",
        origin="lower",
        extent=[time_axis[0], time_axis[-1], 0.5, ifr_view.shape[0] + 0.5],
        cmap="viridis",
        norm=LogNorm(vmin=heatmap_vmin, vmax=heatmap_vmax),
    )
    ax0.set_ylabel("Selected electrode rank")
    ax0.set_yticks([1, ifr_view.shape[0]])
    ax0.set_title(heatmap_title)

    cbar = fig.colorbar(im, cax=cax)
    cbar.set_label("Instantaneous firing rate (Hz, log scale)")
    cbar.locator = LogLocator(base=10)
    cbar.formatter = LogFormatterMathtext(base=10)
    cbar.update_ticks()

    if mean_log_scale:
        ax1.plot(time_axis, np.clip(mean_view, mean_log_floor, None), color="0.70", lw=1.0, label="Population mean IFR")
        ax1.plot(
            time_axis,
            np.clip(mean_smooth_view, mean_log_floor, None),
            color="black",
            lw=2.0,
            label=f"Smoothed mean IFR (sigma={SMOOTH_SIGMA_SEC:.2f} s)",
        )
        ax1.set_yscale("log")
        ax1.set_ylabel("Hz (log scale)")
    else:
        ax1.plot(time_axis, mean_view, color="0.70", lw=1.0, label="Population mean IFR")
        ax1.plot(
            time_axis,
            mean_smooth_view,
            color="black",
            lw=2.0,
            label=f"Smoothed mean IFR (sigma={SMOOTH_SIGMA_SEC:.2f} s)",
        )
        ax1.set_ylabel("Hz")

    ax1.set_xlabel("Time (s)")
    ax1.set_title(mean_title)
    ax1.legend(loc="upper right")
    ax1.set_xlim(float(time_axis[0]), float(time_axis[-1]))
    save_inline_plot(fig, plot_name or heatmap_title)

print(f"IFR matrix shape: {ifr_matrix.shape}")
print(f"Population mean range: {mean_ifr.min():.3f} to {mean_ifr.max():.3f} Hz")
print(f"Smoothed population mean range: {mean_ifr_smooth.min():.3f} to {mean_ifr_smooth.max():.3f} Hz")
global_positive_ifr = ifr_matrix[ifr_matrix > 0]
global_heatmap_vmin = max(1e-3, float(np.quantile(global_positive_ifr, 0.01)))
global_heatmap_vmax = float(global_positive_ifr.max())
print(f"Global heatmap log scale range: {global_heatmap_vmin:.4f} to {global_heatmap_vmax:.4f} Hz")


In [ ]:

plot_ifr_summary(
    time_grid,
    ifr_matrix,
    mean_ifr,
    mean_ifr_smooth,
    heatmap_title="Per-electrode IFR on a shared time grid",
    mean_title="Average firing rate across selected electrodes (log y-scale)",
    mean_log_scale=True,
)


## Zoom Into a High-Activity Window

The full 300 s trace is useful for context, but a short zoom makes the cross-electrode burst structure easier to inspect.


In [ ]:
zoom_center_sec = float(time_grid[np.argmax(mean_ifr_smooth)])
zoom_half_width_sec = 5.0
zoom_start = max(time_grid[0], zoom_center_sec - zoom_half_width_sec)
zoom_stop = min(time_grid[-1], zoom_center_sec + zoom_half_width_sec)
zoom_mask = (time_grid >= zoom_start) & (time_grid <= zoom_stop)

plot_ifr_summary(
    time_grid[zoom_mask],
    ifr_matrix[:, zoom_mask],
    mean_ifr[zoom_mask],
    mean_ifr_smooth[zoom_mask],
    heatmap_title=f"Zoomed IFR matrix: {zoom_start:.2f} s to {zoom_stop:.2f} s",
    mean_title="Zoomed average firing rate across selected electrodes (log y-scale)",
    mean_log_scale=True,
)


## Nested Burst and Gamma Anchors

We now separate the problem into two scales:

- a **coarse burst epoch** on the slow population IFR trace
- a **nested gamma anchor** inside that burst using a high-resolution spike-density signal

This avoids treating several slow subpeaks from the same burst as independent events.


In [ ]:
coarse_epochs, raw_peak_idx, raw_peak_props = detect_coarse_burst_epochs(
    time_grid,
    mean_ifr_smooth,
    grid_hz=IFR_GRID_HZ,
    peak_distance_sec=BURST_MIN_DISTANCE_SEC,
    prominence_quantile=BURST_PROMINENCE_QUANTILE,
    prominence_scale=BURST_PROMINENCE_SCALE,
    rel_height=COARSE_EPOCH_REL_HEIGHT,
)

highres = build_highres_traces(
    rec,
    refs,
    bin_ms=HIGHRES_BIN_MS,
    smooth_sigma_ms=HIGHRES_SMOOTH_SIGMA_MS,
)
highres_data = {
    "bin_edges_s": highres.bin_edges_s,
    "time_centers_s": highres.time_centers_s,
    "electrodes": highres.electrodes,
    "per_electrode_rate_hz": highres.per_electrode_rate_hz,
    "population_rate_hz": highres.population_rate_hz,
    "spikes_by_electrode": highres.spikes_by_electrode,
    "spike_presence": highres.spike_presence,
}

network_activity = build_network_activity_state(
    highres,
    aggregation_ms=NETWORK_BIN_MS,
    active_rate_floor_hz=NETWORK_ACTIVE_RATE_FLOOR_HZ,
    threshold_baseline_quantile=NETWORK_THRESHOLD_BASELINE_QUANTILE,
    threshold_iqr_scale=NETWORK_THRESHOLD_IQR_SCALE,
)
high_activity_epochs, high_activity_info = detect_high_activity_epochs(
    time_grid,
    mean_ifr_smooth,
    mad_scale=HIGH_ACTIVITY_MAD_SCALE,
    min_duration_ms=HIGH_ACTIVITY_MIN_DURATION_MS,
    max_gap_bins=HIGH_ACTIVITY_MAX_GAP_BINS,
)
participation_activity = build_participation_activity_state(
    highres,
    aggregation_ms=NETWORK_BIN_MS,
)
hierarchical_burst_epochs = detect_participation_burst_epochs(
    participation_activity,
    high_activity_epochs,
    min_participation_fraction=NETWORK_MIN_PARTICIPATION_FRACTION,
    min_duration_ms=NETWORK_MIN_DURATION_MS,
)

def assign_max_population_ifr_burst_anchors(highres_traces, burst_epochs):
    if burst_epochs.empty:
        return burst_epochs.copy()

    anchored = burst_epochs.copy()
    if "anchor_time_s" in anchored:
        anchored["participation_anchor_time_s"] = anchored["anchor_time_s"].astype(float)
    else:
        anchored["participation_anchor_time_s"] = np.nan
    anchored["anchor_method"] = "max_highres_population_ifr"

    for idx, row in anchored.iterrows():
        mask = (
            (highres_traces.time_centers_s >= float(row["start_time_s"]))
            & (highres_traces.time_centers_s <= float(row["end_time_s"]))
        )
        local_indices = np.flatnonzero(mask)
        if local_indices.size == 0:
            continue
        local_rate = highres_traces.population_rate_hz[local_indices]
        anchor_idx = int(local_indices[int(np.nanargmax(local_rate))])
        anchored.loc[idx, "anchor_time_s"] = float(highres_traces.time_centers_s[anchor_idx])
        anchored.loc[idx, "coarse_peak_time_s"] = float(highres_traces.time_centers_s[anchor_idx])
        anchored.loc[idx, "coarse_peak_idx"] = anchor_idx
        anchored.loc[idx, "coarse_peak_hz"] = float(highres_traces.population_rate_hz[anchor_idx])
        anchored.loc[idx, "anchor_population_rate_hz"] = float(highres_traces.population_rate_hz[anchor_idx])
    return anchored


hierarchical_burst_epochs = assign_max_population_ifr_burst_anchors(
    highres,
    hierarchical_burst_epochs,
)

analysis_epochs = hierarchical_burst_epochs.copy()
analysis_anchors = analysis_epochs.copy()
if not analysis_anchors.empty:
    analysis_anchors["coarse_event_idx"] = analysis_anchors["event_idx"].astype(int)
    analysis_anchors["gamma_peak_rank"] = 0
    analysis_anchors["onset_time_s"] = analysis_anchors["start_time_s"].astype(float)
    analysis_anchors["anchor_height_hz"] = analysis_anchors["anchor_population_rate_hz"].astype(float)
    analysis_anchors["gamma_anchor_delay_ms"] = (analysis_anchors["anchor_time_s"] - analysis_anchors["onset_time_s"]) * 1000.0
    analysis_anchors["anchor_type"] = "max_highres_population_ifr"

network_coarse_epochs = detect_network_burst_epochs(
    network_activity,
    min_participation_fraction=NETWORK_MIN_PARTICIPATION_FRACTION,
    min_active_electrodes=NETWORK_MIN_ACTIVE_ELECTRODES,
    merge_gap_ms=NETWORK_MERGE_GAP_MS,
    min_duration_ms=NETWORK_MIN_DURATION_MS,
    min_spikes=NETWORK_MIN_SPIKES,
)
network_nested_anchors = detect_nested_gamma_anchors(
    network_coarse_epochs,
    highres,
    coarse_rel_height=COARSE_EPOCH_REL_HEIGHT,
    bin_ms=HIGHRES_BIN_MS,
    search_ms=GAMMA_SEARCH_MS,
    search_to_epoch_end=GAMMA_SEARCH_TO_EPOCH_END,
    min_distance_ms=GAMMA_MIN_DISTANCE_MS,
    prominence_frac=GAMMA_PROMINENCE_FRAC,
    prominence_abs_floor=GAMMA_PROMINENCE_ABS_FLOOR,
    keep_height_frac=GAMMA_KEEP_HEIGHT_FRAC,
)

nested_anchors = detect_nested_gamma_anchors(
    coarse_epochs,
    highres,
    coarse_rel_height=COARSE_EPOCH_REL_HEIGHT,
    bin_ms=HIGHRES_BIN_MS,
    search_ms=GAMMA_SEARCH_MS,
    search_to_epoch_end=GAMMA_SEARCH_TO_EPOCH_END,
    min_distance_ms=GAMMA_MIN_DISTANCE_MS,
    prominence_frac=GAMMA_PROMINENCE_FRAC,
    prominence_abs_floor=GAMMA_PROMINENCE_ABS_FLOOR,
    keep_height_frac=GAMMA_KEEP_HEIGHT_FRAC,
)

print(f"Raw slow peaks: {len(raw_peak_idx)}")
print(f"Rate-peak coarse burst epochs: {len(coarse_epochs)}")
print(f"Participation-gated coarse burst epochs: {len(network_coarse_epochs)}")
print(f"High-activity periods: {len(high_activity_epochs)}")
print(f"Nested high-activity participation bursts: {len(hierarchical_burst_epochs)}")
print(f"High-activity threshold: {high_activity_info['threshold_hz']:.3f} Hz")
print(f"Rate-peak nested gamma anchors: {len(nested_anchors)}")
print(f"Participation-gated nested gamma anchors: {len(network_nested_anchors)}")

comparison_summary = pd.DataFrame(
    [
        {
            "method": "rate_peak",
            "n_macro_bursts": len(coarse_epochs),
            "n_burst_peak_anchors": len(coarse_epochs),
            "n_gamma_anchors": len(nested_anchors),
            "anchor_type": "rate_peak_plus_nested_gamma",
            "total_macro_duration_s": float((coarse_epochs["end_time_s"] - coarse_epochs["start_time_s"]).sum()) if not coarse_epochs.empty else 0.0,
        },
        {
            "method": "network_participation",
            "n_macro_bursts": len(network_coarse_epochs),
            "n_burst_peak_anchors": len(network_coarse_epochs),
            "n_gamma_anchors": len(network_nested_anchors),
            "anchor_type": "participation_epoch_plus_nested_gamma",
            "total_macro_duration_s": float((network_coarse_epochs["end_time_s"] - network_coarse_epochs["start_time_s"]).sum()) if not network_coarse_epochs.empty else 0.0,
        },
        {
            "method": "high_activity_participation",
            "n_macro_bursts": len(analysis_epochs),
            "n_burst_peak_anchors": len(analysis_anchors),
            "n_gamma_anchors": np.nan,
            "anchor_type": "max_highres_population_ifr",
            "total_macro_duration_s": float((analysis_epochs["end_time_s"] - analysis_epochs["start_time_s"]).sum()) if not analysis_epochs.empty else 0.0,
        },
    ]
)
display(comparison_summary.round(3))

display(analysis_anchors[[
    "event_idx",
    "high_activity_event_idx",
    "start_time_s",
    "end_time_s",
    "anchor_time_s",
    "participation_anchor_time_s",
    "anchor_participation_fraction",
    "anchor_population_rate_hz",
    "peak_active_electrodes",
    "participating_electrodes",
    "duration_ms",
    "anchor_type",
]].head(10).round(3))


In [ ]:
fig, axes = plot_high_activity_burst_windows(
    time_grid=time_grid,
    mean_ifr=mean_ifr,
    mean_ifr_smooth=mean_ifr_smooth,
    network_activity=participation_activity,
    high_activity_epochs=high_activity_epochs,
    burst_epochs=hierarchical_burst_epochs,
    high_activity_threshold_hz=high_activity_info["threshold_hz"],
    participation_threshold=NETWORK_MIN_PARTICIPATION_FRACTION,
    n_windows=3,
    pad_s=max(1.5, float(MACRO_BURST_ZOOM_PAD_SEC)),
)
fig.suptitle(
    "High-activity periods with nested burst anchors",
    y=1.02,
)
save_inline_plot(fig, "high_activity_burst_windows")

display(high_activity_epochs.head(10).round(3))
display(hierarchical_burst_epochs[[
    "event_idx",
    "high_activity_event_idx",
    "start_time_s",
    "end_time_s",
    "anchor_time_s",
    "participation_anchor_time_s",
    "anchor_participation_fraction",
    "anchor_population_rate_hz",
    "peak_active_electrodes",
    "participating_electrodes",
    "total_spikes",
    "duration_ms",
]].head(12).round(3))


## Burst-Window IFR Summary
Use the shared `plot_ifr_summary` view for representative windows that were designated as bursts. The burst span and refined max-participation anchor are overlaid on the population summary panel.


In [ ]:
if analysis_epochs.empty:
    raise ValueError("No hierarchical participation bursts were detected for burst-window summaries.")

burst_event_ids = (
    analysis_epochs.sort_values("anchor_population_rate_hz", ascending=False)["event_idx"]
    .head(int(MACRO_BURST_ZOOM_COUNT))
    .to_numpy(dtype=int)
)
burst_event_ids = np.sort(burst_event_ids)

for event_id in burst_event_ids:
    burst_row = analysis_epochs.loc[analysis_epochs["event_idx"] == int(event_id)].iloc[0]
    view_start = max(float(time_grid[0]), float(burst_row["start_time_s"]) - float(MACRO_BURST_ZOOM_PAD_SEC))
    view_stop = min(float(time_grid[-1]), float(burst_row["end_time_s"]) + float(MACRO_BURST_ZOOM_PAD_SEC))
    view_mask = (time_grid >= view_start) & (time_grid <= view_stop)
    if int(np.sum(view_mask)) < 2:
        continue

    plot_ifr_summary(
        time_grid[view_mask],
        ifr_matrix[:, view_mask],
        mean_ifr[view_mask],
        mean_ifr_smooth[view_mask],
        heatmap_title=f"Burst-window IFR matrix: event {int(event_id)} ({view_start:.2f}-{view_stop:.2f} s)",
        mean_title=(
            f"Designated burst window {int(event_id)}: "
            f"{float(burst_row['start_time_s']):.3f}-{float(burst_row['end_time_s']):.3f} s"
        ),
        mean_log_scale=True,
    )


## IFR Histogram by Activity State
Split raw ISI-derived IFR samples into low-activity, high-activity non-burst, and burst periods. The histogram uses stacked bars so each IFR bin shows how much each activity state contributes to the same IFR range.


In [ ]:
state_ifr_positive = extract_activity_state_ifr(
    rec,
    refs,
    high_activity_epochs,
    analysis_epochs,
    max_hz=IFR_MAX_HZ,
)
for state, vals in state_ifr_positive.items():
    if vals.size == 0:
        raise ValueError(f"No positive raw IFR values were found for state: {state}")

all_state_ifr = np.concatenate(list(state_ifr_positive.values()))
hist_floor_hz = max(1e-3, float(np.min(all_state_ifr)))
hist_ceiling_hz = float(np.max(all_state_ifr))
if np.isclose(hist_floor_hz, hist_ceiling_hz):
    hist_ceiling_hz = hist_floor_hz * 10.0
hist_bins_hz = np.logspace(np.log10(hist_floor_hz), np.log10(hist_ceiling_hz), 80)

state_order = ["low_activity", "high_activity", "burst"]
state_labels = {
    "low_activity": "Low activity",
    "high_activity": "High activity, non-burst",
    "burst": "Burst",
}
state_colors = {
    "low_activity": "0.55",
    "high_activity": "tab:green",
    "burst": "tab:blue",
}
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
for state in state_order:
    axes[0].hist(
        state_ifr_positive[state],
        bins=hist_bins_hz,
        color=state_colors[state],
        alpha=0.42,
        edgecolor="white",
        linewidth=0.25,
        label=f"{state_labels[state]} (n={state_ifr_positive[state].size:,})",
    )
axes[0].set_xscale("log")
axes[0].set_xlabel("Instantaneous firing rate (Hz)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Raw IFR by activity state, overlaid")
axes[0].legend(loc="upper right")
axes[0].grid(True, axis="x", alpha=0.18)

quantile_grid = np.linspace(0.0, 1.0, 201)
for state in state_order:
    axes[1].plot(
        quantile_grid,
        np.quantile(state_ifr_positive[state], quantile_grid),
        color=state_colors[state],
        lw=2.0,
        label=state_labels[state],
    )
axes[1].set_yscale("log")
axes[1].set_xlabel("Quantile")
axes[1].set_ylabel("Instantaneous firing rate (Hz)")
axes[1].set_title("Raw-ISI IFR quantile comparison")
axes[1].legend(loc="upper left")
axes[1].grid(True, which="both", axis="y", alpha=0.18)
save_inline_plot(fig, "single_recording_ifr_by_activity_state")

summary_rows = []
for state in state_order:
    vals = state_ifr_positive[state]
    summary_rows.append(
        {
            "period": state,
            "n_samples": int(vals.size),
            "median_ifr_hz": float(np.median(vals)),
            "mean_ifr_hz": float(np.mean(vals)),
            "p90_ifr_hz": float(np.quantile(vals, 0.90)),
            "p99_ifr_hz": float(np.quantile(vals, 0.99)),
        }
    )
burst_ifr_positive = state_ifr_positive["burst"]
pd.DataFrame(summary_rows).round(3)


## Aggregate IFR Histogram by Activity State Across Wells
Run the same low-activity / high-activity / burst split for each requested well and DIV, then pool raw ISI-derived IFR samples by activity state for an aggregate histogram.


In [ ]:
aggregate_state_chunks = {state: [] for state in state_order}
aggregate_rows = []
aggregate_prep_cfg = PrepConfig(mode="top", top_start=TOP_START, top_stop=TOP_STOP, verbose=False)

for spec_i in iter_aggregate_recording_specs():
    ds_i = load_recording_from_spec(spec_i)
    rec_i = ds_i.recordings[0]
    refs_i = ds_i.select_ref_electrodes(aggregate_prep_cfg)[0]
    population_i = build_population_ifr(rec_i, refs_i, grid_hz=IFR_GRID_HZ, smooth_sigma_sec=SMOOTH_SIGMA_SEC)
    highres_i = build_highres_traces(rec_i, refs_i, bin_ms=HIGHRES_BIN_MS, smooth_sigma_ms=HIGHRES_SMOOTH_SIGMA_MS)
    high_epochs_i, high_info_i = detect_high_activity_epochs(
        population_i.time_grid,
        population_i.mean_ifr_smooth,
        mad_scale=HIGH_ACTIVITY_MAD_SCALE,
        min_duration_ms=HIGH_ACTIVITY_MIN_DURATION_MS,
        max_gap_bins=HIGH_ACTIVITY_MAX_GAP_BINS,
    )
    participation_i = build_participation_activity_state(highres_i, aggregation_ms=NETWORK_BIN_MS)
    burst_epochs_i = detect_participation_burst_epochs(
        participation_i,
        high_epochs_i,
        min_participation_fraction=NETWORK_MIN_PARTICIPATION_FRACTION,
        min_duration_ms=NETWORK_MIN_DURATION_MS,
    )
    burst_epochs_i = assign_max_population_ifr_burst_anchors(
        highres_i,
        burst_epochs_i,
    )
    state_chunks_i = extract_activity_state_ifr(rec_i, refs_i, high_epochs_i, burst_epochs_i, max_hz=IFR_MAX_HZ)
    state_counts_i = {}
    for state in state_order:
        vals = np.asarray(state_chunks_i[state], dtype=float)
        aggregate_state_chunks[state].append(vals)
        state_counts_i[f"n_{state}_ifr_samples"] = int(vals.size)

    aggregate_rows.append(
        {
            "well": int(spec_i["well"]),
            "div": spec_i.get("div", np.nan),
            "recording_id": spec_i["recording_id"],
            "dataset": spec_i["dataset"],
            "source_file": spec_i["source_file"],
            "n_refs": int(len(refs_i)),
            "n_high_activity_periods": int(len(high_epochs_i)),
            "n_bursts": int(len(burst_epochs_i)),
            "high_activity_threshold_hz": float(high_info_i["threshold_hz"]),
            "total_high_activity_duration_s": float((high_epochs_i["end_time_s"] - high_epochs_i["start_time_s"]).sum()) if not high_epochs_i.empty else 0.0,
            "total_burst_duration_s": float((burst_epochs_i["end_time_s"] - burst_epochs_i["start_time_s"]).sum()) if not burst_epochs_i.empty else 0.0,
            **state_counts_i,
        }
    )

aggregate_activity_summary = pd.DataFrame(aggregate_rows)
if aggregate_activity_summary.empty:
    raise ValueError("No aggregate recordings were processed. Check AGGREGATE_WELLS and AGGREGATE_DIVS.")

aggregate_state_ifr_positive = {}
for state in state_order:
    vals = np.concatenate([chunk for chunk in aggregate_state_chunks[state] if chunk.size > 0]).astype(float) if aggregate_state_chunks[state] else np.array([], dtype=float)
    vals = vals[np.isfinite(vals) & (vals > 0) & (vals <= IFR_MAX_HZ)]
    if vals.size == 0:
        raise ValueError(f"No aggregate positive raw IFR values were found for state: {state}")
    aggregate_state_ifr_positive[state] = vals

aggregate_all_ifr = np.concatenate(list(aggregate_state_ifr_positive.values()))
aggregate_hist_floor_hz = max(1e-3, float(np.min(aggregate_all_ifr)))
aggregate_hist_ceiling_hz = float(np.max(aggregate_all_ifr))
if np.isclose(aggregate_hist_floor_hz, aggregate_hist_ceiling_hz):
    aggregate_hist_ceiling_hz = aggregate_hist_floor_hz * 10.0
aggregate_hist_bins_hz = np.logspace(np.log10(aggregate_hist_floor_hz), np.log10(aggregate_hist_ceiling_hz), 90)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
for state in state_order:
    axes[0].hist(
        aggregate_state_ifr_positive[state],
        bins=aggregate_hist_bins_hz,
        color=state_colors[state],
        alpha=0.40,
        edgecolor="white",
        linewidth=0.25,
        label=f"{state_labels[state]} (n={aggregate_state_ifr_positive[state].size:,})",
    )
axes[0].set_xscale("log")
axes[0].set_xlabel("Instantaneous firing rate (Hz)")
axes[0].set_ylabel("Frequency")
axes[0].set_title(f"Aggregate raw IFR by activity state | dataset={DATASET}, wells={AGGREGATE_WELLS}")
axes[0].legend(loc="upper left")
axes[0].grid(True, axis="x", alpha=0.18)

for state in state_order:
    axes[1].plot(
        quantile_grid,
        np.quantile(aggregate_state_ifr_positive[state], quantile_grid),
        color=state_colors[state],
        lw=2.0,
        label=state_labels[state],
    )
axes[1].set_yscale("log")
axes[1].set_xlabel("Quantile")
axes[1].set_ylabel("Instantaneous firing rate (Hz)")
axes[1].set_title("Aggregate raw-ISI IFR quantile comparison")
axes[1].legend(loc="upper left")
axes[1].grid(True, which="both", axis="y", alpha=0.18)
save_inline_plot(fig, "aggregate_ifr_by_activity_state")

aggregate_activity_by_div = (
    aggregate_activity_summary
    .groupby("div", as_index=False)
    .agg(
        n_recordings=("recording_id", "count"),
        n_wells=("well", "nunique"),
        mean_high_activity_periods=("n_high_activity_periods", "mean"),
        mean_bursts=("n_bursts", "mean"),
        total_high_activity_duration_s=("total_high_activity_duration_s", "sum"),
        total_burst_duration_s=("total_burst_duration_s", "sum"),
        total_low_activity_ifr_samples=("n_low_activity_ifr_samples", "sum"),
        total_high_activity_ifr_samples=("n_high_activity_ifr_samples", "sum"),
        total_burst_ifr_samples=("n_burst_ifr_samples", "sum"),
    )
)

display(aggregate_activity_summary.round(3))
display(aggregate_activity_by_div.round(3))


## Aggregate High-Activity vs Burst IFR Histograms with Binned-KDE Maxima

Compare raw-ISI IFR values from high-activity non-burst periods and burst periods pooled across `AGGREGATE_WELLS` for the configured dataset. For `stim_removal_null`, this uses the selected `AGGREGATE_DIVS` list, usually one DIV for same-DIV well aggregation.


In [ ]:
activity_kde_source_label = f"aggregate {DATASET} wells={AGGREGATE_WELLS}"
if DATASET == "stim_removal_null":
    activity_kde_source_label += f", DIVs={AGGREGATE_DIVS}"

activity_kde_inputs = {
    "high_activity": aggregate_state_ifr_positive["high_activity"],
    "burst": aggregate_state_ifr_positive["burst"],
}
activity_kde_results = {
    state: binned_kde_peak_summary(
        values,
        log_bins=True,
        n_bins=260,
        grid_size=8192,
        prominence_fraction=0.012,
        distance_fraction=0.006,
        bandwidth_scale=0.22,
        max_hz=IFR_MAX_HZ,
    )
    for state, values in activity_kde_inputs.items()
}
activity_ifr_frequency_values_hz = activity_state_kde_peak_frequencies(
    activity_kde_results,
    states=("high_activity", "burst"),
    min_peak_hz=30.0,
)

fig, axes = plot_activity_state_ifr_kde_histograms(
    activity_kde_results,
    activity_kde_inputs,
    states=("high_activity", "burst"),
    state_labels=state_labels,
    state_colors=state_colors,
    source_label=activity_kde_source_label,
)
save_inline_plot(fig, "aggregate_activity_state_ifr_binned_kde_maxima")

activity_ifr_kde_peak_summary = pd.concat(
    [
        pd.DataFrame(
            {
                "period": state,
                "source": activity_kde_source_label,
                "peak_hz": activity_kde_results[state]["peak_hz"],
                "peak_count": activity_kde_results[state]["peak_counts"],
                "used_for_distance_model": activity_kde_results[state]["peak_hz"] > 30.0,
            }
        )
        for state in ["high_activity", "burst"]
    ],
    ignore_index=True,
).sort_values(["period", "peak_count"], ascending=[True, False], ignore_index=True)

print("KDE source:", activity_kde_source_label)
print("High-activity binned-KDE peaks:", np.round(activity_kde_results["high_activity"]["peak_hz"][:6], 3))
print("Burst binned-KDE peaks:", np.round(activity_kde_results["burst"]["peak_hz"][:6], 3))
print("Merged distance-model frequencies >30 Hz:", np.round(activity_ifr_frequency_values_hz, 3))
activity_ifr_kde_peak_summary.head(12)


In [ ]:
aligned = align_highres_to_anchors(
    highres,
    analysis_anchors,
    pre_ms=ALIGN_PRE_MS,
    post_ms=ALIGN_POST_MS,
    bin_ms=HIGHRES_BIN_MS,
)

pre_bins = int(round(ALIGN_PRE_MS / HIGHRES_BIN_MS))
post_bins = int(round(ALIGN_POST_MS / HIGHRES_BIN_MS))
relative_time_ms = aligned.relative_time_ms

time_centers_highres = highres.time_centers_s
population_rate_highres = highres.population_rate_hz
per_electrode_rate_highres = highres.per_electrode_rate_hz
selected_electrodes = highres.electrodes
spikes_by_electrode = highres.spikes_by_electrode
spike_presence_highres = highres.spike_presence
population_windows = aligned.population_windows
aligned_tensor = aligned.aligned_rate
aligned_spike_tensor = aligned.aligned_spikes
valid_burst_peak_anchors = pd.DataFrame(aligned.valid_anchors)
valid_nested_anchors = valid_burst_peak_anchors


def select_example_events(anchors_df, count):
    if anchors_df is None or len(anchors_df) == 0:
        return pd.DataFrame()
    count = min(int(count), len(anchors_df))
    if "anchor_height_hz" in anchors_df:
        anchor_heights = anchors_df["anchor_height_hz"].to_numpy(dtype=float)
    elif "anchor_population_rate_hz" in anchors_df:
        anchor_heights = anchors_df["anchor_population_rate_hz"].to_numpy(dtype=float)
    else:
        anchor_heights = np.arange(len(anchors_df), dtype=float)
    sorted_indices = np.argsort(anchor_heights)

    highest_candidates = [int(idx) for idx in sorted_indices[::-1][:2]]
    median_center = len(sorted_indices) // 2
    median_start = max(0, median_center - 1)
    median_stop = min(len(sorted_indices), median_start + 2)
    median_candidates = [int(idx) for idx in sorted_indices[median_start:median_stop]]
    lowest_candidates = [int(idx) for idx in sorted_indices[:2]]

    example_indices = []
    selection_labels = {}
    for band_name, candidates in [("highest", highest_candidates), ("median", median_candidates), ("lowest", lowest_candidates)]:
        for idx in candidates:
            if idx not in example_indices:
                example_indices.append(idx)
                selection_labels[idx] = band_name

    if len(example_indices) < count:
        for idx in sorted_indices[::-1]:
            idx = int(idx)
            if idx not in example_indices:
                example_indices.append(idx)
                selection_labels[idx] = "additional"
            if len(example_indices) >= count:
                break

    example_indices = np.asarray(example_indices[:count], dtype=int)
    out = anchors_df.iloc[example_indices].copy()
    out["window_idx"] = example_indices
    out["selection_band"] = [selection_labels[int(idx)] for idx in example_indices]
    out["selection_band"] = pd.Categorical(
        out["selection_band"],
        categories=["highest", "median", "lowest", "additional"],
        ordered=True,
    )
    if "gamma_peak_rank" not in out:
        out["gamma_peak_rank"] = 0
    sort_cols = [col for col in ["selection_band", "coarse_event_idx", "gamma_peak_rank"] if col in out]
    return out.sort_values(sort_cols).reset_index(drop=True)


example_events = select_example_events(valid_nested_anchors, EXAMPLE_GAMMA_EVENT_COUNT)

fig, ax = plt.subplots(figsize=(11, 5))
for row in population_windows:
    ax.plot(relative_time_ms, row, color="0.78", alpha=0.35)
ax.plot(relative_time_ms, population_windows.mean(axis=0), color="black", lw=2.5, label="Mean burst-peak-centered population trace")
ax.axvline(0.0, color="crimson", ls="--", lw=1.2, label="Refined burst peak anchor")
ax.set_xlabel("Time relative to burst peak anchor (ms)")
ax.set_ylabel("Population spike-density rate (Hz)")
ax.set_title(f"Burst-peak-centered population windows ({len(valid_nested_anchors)} events)")
ax.legend(loc="upper right")
plt.tight_layout()
save_inline_plot(fig, "burst_peak_centered_population_windows")

print(f"Valid burst-peak-centered windows: {len(valid_nested_anchors)}")
print(f"Aligned tensor shape: {aligned_tensor.shape}")


## Signed Eventwise X-Bin Wave-Peak Analysis
Replace the earlier signed-bin TE-like contrast with a simpler eventwise propagation summary. For each burst-peak-centered event, bin electrodes by **signed x-position** relative to the array midpoint, average the per-electrode spike-density traces within each x-bin, and record the **time of peak activity** for each bin relative to the event peak.

This gives a direct macroscopic wave readout: if the average peak time drifts systematically with signed x-position, the fitted slope of `delay vs signed x` yields an implied propagation speed for the wave peak. The plots below show the mean x-binned traces, the eventwise peak-delay cloud with bin-level summaries, and a bootstrap confidence interval for the implied speed.


In [ ]:
if len(valid_nested_anchors) == 0:
    raise ValueError("No valid burst-peak-centered windows are available for signed eventwise wave analysis.")


def wave_cache_key():
    return (
        f"xbin{float(WAVE_X_BIN_UM):g}_"
        f"peak{float(WAVE_PEAK_SEARCH_START_MS):g}to{float(WAVE_PEAK_SEARCH_STOP_MS):g}_"
        f"smooth{float(WAVE_TRACE_SMOOTH_SIGMA_MS):g}_"
        f"bin{float(HIGHRES_BIN_MS):g}_"
        f"minelec{int(WAVE_MIN_ELECTRODES_PER_BIN)}_"
        f"minevents{int(WAVE_MIN_EVENTS_PER_BIN)}_"
        f"boot{int(WAVE_BOOTSTRAP_REPS)}_seed{int(WAVE_RANDOM_SEED)}"
    ).replace(".", "p").replace("-", "m")


def wave_cache_dir(recording_id):
    return Path(WAVE_CACHE_ROOT) / str(recording_id) / wave_cache_key()


def save_wave_result_cache(result, cache_dir):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    result.event_direction.to_csv(cache_dir / "event_direction.csv", index=False)
    result.trace.to_csv(cache_dir / "trace.csv", index=False)
    result.peaks.to_csv(cache_dir / "peaks.csv", index=False)
    result.bin_summary.to_csv(cache_dir / "bin_summary.csv", index=False)
    result.fit_summary.to_csv(cache_dir / "fit_summary.csv", index=False)
    pd.DataFrame(result.heatmap).to_csv(cache_dir / "heatmap.csv", index=True, index_label="time_ms")
    pd.DataFrame({"speed_um_per_ms": result.bootstrap_speeds}).to_csv(cache_dir / "bootstrap_speeds.csv", index=False)
    return cache_dir


def load_wave_result_cache(cache_dir):
    cache_dir = Path(cache_dir)
    required = [
        "event_direction.csv",
        "trace.csv",
        "peaks.csv",
        "bin_summary.csv",
        "fit_summary.csv",
        "heatmap.csv",
        "bootstrap_speeds.csv",
    ]
    if not all((cache_dir / name).exists() for name in required):
        return None
    heatmap = pd.read_csv(cache_dir / "heatmap.csv", index_col=0)
    heatmap.index = heatmap.index.astype(float)
    heatmap.columns = heatmap.columns.astype(float)
    boot_df = pd.read_csv(cache_dir / "bootstrap_speeds.csv")
    bootstrap_speeds = boot_df["speed_um_per_ms"].to_numpy(dtype=float)
    fit_summary = pd.read_csv(cache_dir / "fit_summary.csv")
    if "bootstrap_speed_mean_um_per_ms" not in fit_summary.columns:
        fit_summary["bootstrap_speed_mean_um_per_ms"] = float(np.mean(bootstrap_speeds)) if bootstrap_speeds.size else np.nan
    return WaveAnalysisResult(
        event_direction=pd.read_csv(cache_dir / "event_direction.csv"),
        trace=pd.read_csv(cache_dir / "trace.csv"),
        peaks=pd.read_csv(cache_dir / "peaks.csv"),
        bin_summary=pd.read_csv(cache_dir / "bin_summary.csv"),
        fit_summary=fit_summary,
        heatmap=heatmap,
        bootstrap_speeds=bootstrap_speeds,
    )


def compute_or_load_wave_result(recording_id, aligned_events, layout, *, force=False):
    cache_dir = wave_cache_dir(recording_id)
    if not force:
        cached = load_wave_result_cache(cache_dir)
        if cached is not None:
            print(f"Loaded cached wave result: {cache_dir}")
            return cached

    result = analyze_eventwise_waves(
        aligned_events,
        layout,
        x_bin_um=WAVE_X_BIN_UM,
        peak_search_start_ms=WAVE_PEAK_SEARCH_START_MS,
        peak_search_stop_ms=WAVE_PEAK_SEARCH_STOP_MS,
        trace_smooth_sigma_ms=WAVE_TRACE_SMOOTH_SIGMA_MS,
        bin_ms=HIGHRES_BIN_MS,
        min_electrodes_per_bin=WAVE_MIN_ELECTRODES_PER_BIN,
        min_events_per_bin=WAVE_MIN_EVENTS_PER_BIN,
        bootstrap_reps=WAVE_BOOTSTRAP_REPS,
        random_seed=WAVE_RANDOM_SEED,
    )
    save_wave_result_cache(result, cache_dir)
    print(f"Saved wave result cache: {cache_dir}")
    return result


def plot_wave_result_figure(result, *, title_prefix="", show_tables=True):
    wave_event_direction_df = result.event_direction
    wave_peak_df = result.peaks
    wave_bin_summary = result.bin_summary
    wave_peak_fit_summary = result.fit_summary.copy()
    wave_heatmap = result.heatmap
    wave_boot_speeds = result.bootstrap_speeds

    if "bootstrap_speed_mean_um_per_ms" not in wave_peak_fit_summary.columns:
        wave_peak_fit_summary["bootstrap_speed_mean_um_per_ms"] = float(np.mean(wave_boot_speeds)) if wave_boot_speeds.size else np.nan

    wave_fit_row = wave_peak_fit_summary.iloc[0]
    wave_array_width_um = float(wave_fit_row["array_width_um"])
    wave_fit_slope_ms_per_um = float(wave_fit_row["slope_ms_per_um"])
    wave_fit_intercept_ms = float(wave_fit_row["intercept_ms"])
    wave_speed_um_per_ms = float(wave_fit_row["implied_speed_um_per_ms"])
    wave_speed_boot_mean = float(wave_fit_row["bootstrap_speed_mean_um_per_ms"])
    wave_speed_boot_median = float(wave_fit_row["bootstrap_speed_median_um_per_ms"])
    wave_speed_ci_low = float(wave_fit_row["bootstrap_speed_ci_low_um_per_ms"])
    wave_speed_ci_high = float(wave_fit_row["bootstrap_speed_ci_high_um_per_ms"])
    wave_unique_events = np.sort(wave_peak_df["window_idx"].unique())
    wave_direction_counts = wave_event_direction_df["event_direction"].value_counts().reindex(["left_to_right", "right_to_left"], fill_value=0)
    wave_heatmap_time_ms = wave_heatmap.index.to_numpy(dtype=float)
    wave_heatmap_x_um = wave_heatmap.columns.to_numpy(dtype=float)
    wave_heatmap_values = wave_heatmap.to_numpy(dtype=float)
    wave_heatmap_peaks = wave_bin_summary.set_index("origin_x_um").reindex(wave_heatmap.columns)

    fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), constrained_layout=True)
    ax0, ax1 = axes

    heat_extent = [
        float(max(0.0, wave_heatmap_x_um.min() - 0.5 * WAVE_X_BIN_UM)),
        float(min(wave_array_width_um, wave_heatmap_x_um.max() + 0.5 * WAVE_X_BIN_UM)),
        float(wave_heatmap_time_ms.min()),
        float(wave_heatmap_time_ms.max()),
    ]
    im0 = ax0.imshow(
        wave_heatmap_values,
        aspect="auto",
        origin="lower",
        extent=heat_extent,
        cmap="viridis",
        alpha=0.95,
    )
    ax0.axhline(0.0, color="white", ls="--", lw=1.0)
    ax0.errorbar(
        wave_bin_summary["origin_x_um"],
        wave_bin_summary["mean_peak_time_ms"],
        yerr=1.96 * wave_bin_summary["sem_peak_time_ms"],
        fmt="o",
        color="black",
        capsize=4,
        lw=1.3,
        label="Mean +/-95% CI",
    )
    x_line = np.linspace(float(wave_bin_summary["origin_x_um"].min()), float(wave_bin_summary["origin_x_um"].max()), 200)
    ax0.plot(x_line, wave_fit_slope_ms_per_um * x_line + wave_fit_intercept_ms, color="crimson", ls="--", lw=2.0, label="Linear fit")
    ax0.set_xlim(0.0, wave_array_width_um)
    ax0.set_xlabel("Distance from inferred origin side (um)")
    ax0.set_ylabel("Peak time relative to event peak (ms)")
    wave_direction_counts_text = (
        f"L->R: {int(wave_direction_counts['left_to_right'])} | "
        f"R->L: {int(wave_direction_counts['right_to_left'])}"
    )
    ax0.set_title(
        (
            f"{title_prefix}Wave-peak timing on origin-aligned traces\n"
            f"speed ~ {wave_speed_um_per_ms:.0f} um/ms | {wave_direction_counts_text}"
        )
        if np.isfinite(wave_speed_um_per_ms)
        else f"{title_prefix}Wave-peak timing on origin-aligned traces\n{wave_direction_counts_text}"
    )
    ax0.legend(loc="upper left", fontsize=9)
    fig.colorbar(im0, ax=ax0, label="Mean spike-density rate (Hz)")

    if wave_boot_speeds.size:
        ax1.hist(wave_boot_speeds, bins=30, color="0.75", edgecolor="0.35")
        ax1.axvline(wave_speed_um_per_ms, color="crimson", lw=2.0, label=f"Linear Fit: {wave_speed_um_per_ms:.0f} um/ms")
        ax1.axvline(wave_speed_boot_mean, color="black", lw=1.7, ls="--", label=f"Bootstrap mean: {wave_speed_boot_mean:.0f} um/ms")
        ax1.axvspan(wave_speed_ci_low, wave_speed_ci_high, color="gold", alpha=0.25, label=f"95% CI: [{wave_speed_ci_low:.0f}, {wave_speed_ci_high:.0f}]")
        ax1.set_title(f"{title_prefix}Bootstrap implied speed distribution")
        ax1.set_xlabel("Implied propagation speed (um/ms)")
        ax1.set_ylabel("Bootstrap count")
        ax1.legend(loc="upper right", fontsize=9)
    else:
        ax1.text(0.5, 0.5, "Bootstrap speed distribution unavailable", ha="center", va="center", transform=ax1.transAxes)
        ax1.set_axis_off()

    save_inline_plot(fig, f"{title_prefix}wave_peak_timing_origin_aligned")
    print(
        f"{title_prefix}Signed eventwise wave analysis: {len(wave_unique_events)} reoriented events | "
        f"{len(wave_bin_summary)} populated origin-referenced x-bins | x-bin width={WAVE_X_BIN_UM:.0f} um"
    )
    if show_tables:
        display(wave_peak_fit_summary.round(3))
        display(wave_bin_summary.round(3))
        display(wave_event_direction_df.round(6))
    return fig, axes


recording_id = f"stimRemovalNull_well{int(WELL)}_DIV{int(DIV)}"
wave_result = compute_or_load_wave_result(
    recording_id,
    aligned,
    rec.layout,
    force=bool(WAVE_FORCE_RECOMPUTE),
)

wave_event_direction_df = wave_result.event_direction
wave_trace_df = wave_result.trace
wave_peak_df = wave_result.peaks
wave_bin_summary = wave_result.bin_summary
wave_peak_fit_summary = wave_result.fit_summary
wave_heatmap = wave_result.heatmap
wave_boot_speeds = wave_result.bootstrap_speeds

plot_wave_result_figure(wave_result)


## Aggregate Signed Eventwise X-Bin Wave-Peak Analysis Across Wells
Compute or load one cached wave result per requested well and DIV, then pool eventwise wave peaks across recordings for an aggregate propagation figure.


In [ ]:
def build_aligned_for_recording(recording, selected_refs):
    population_i = build_population_ifr(recording, selected_refs, grid_hz=IFR_GRID_HZ, smooth_sigma_sec=SMOOTH_SIGMA_SEC)
    highres_i = build_highres_traces(recording, selected_refs, bin_ms=HIGHRES_BIN_MS, smooth_sigma_ms=HIGHRES_SMOOTH_SIGMA_MS)
    high_epochs_i, high_info_i = detect_high_activity_epochs(
        population_i.time_grid,
        population_i.mean_ifr_smooth,
        mad_scale=HIGH_ACTIVITY_MAD_SCALE,
        min_duration_ms=HIGH_ACTIVITY_MIN_DURATION_MS,
        max_gap_bins=HIGH_ACTIVITY_MAX_GAP_BINS,
    )
    participation_i = build_participation_activity_state(highres_i, aggregation_ms=NETWORK_BIN_MS)
    burst_epochs_i = detect_participation_burst_epochs(
        participation_i,
        high_epochs_i,
        min_participation_fraction=NETWORK_MIN_PARTICIPATION_FRACTION,
        min_duration_ms=NETWORK_MIN_DURATION_MS,
    )
    burst_epochs_i = assign_max_population_ifr_burst_anchors(highres_i, burst_epochs_i)
    anchors_i = burst_epochs_i.copy()
    if not anchors_i.empty:
        anchors_i["coarse_event_idx"] = anchors_i["event_idx"].astype(int)
        anchors_i["gamma_peak_rank"] = 0
        anchors_i["onset_time_s"] = anchors_i["start_time_s"].astype(float)
        anchors_i["anchor_height_hz"] = anchors_i["anchor_population_rate_hz"].astype(float)
        anchors_i["gamma_anchor_delay_ms"] = (anchors_i["anchor_time_s"] - anchors_i["onset_time_s"]) * 1000.0
        anchors_i["anchor_type"] = "max_highres_population_ifr"
    aligned_i = align_highres_to_anchors(
        highres_i,
        anchors_i,
        pre_ms=ALIGN_PRE_MS,
        post_ms=ALIGN_POST_MS,
        bin_ms=HIGHRES_BIN_MS,
    )
    return aligned_i, highres_i, high_epochs_i, burst_epochs_i, high_info_i


def aggregate_wave_results(results_by_recording):
    peak_frames = []
    trace_frames = []
    direction_frames = []
    fit_frames = []
    event_offset = 0
    for recording_id_i, result_i in results_by_recording:
        peaks_i = result_i.peaks.copy()
        trace_i = result_i.trace.copy()
        directions_i = result_i.event_direction.copy()
        fit_i = result_i.fit_summary.copy()

        for df in (peaks_i, trace_i, directions_i):
            df["recording_id"] = recording_id_i
            if "window_idx" in df:
                df["local_window_idx"] = df["window_idx"].astype(int)
                df["window_idx"] = df["local_window_idx"] + int(event_offset)
        fit_i["recording_id"] = recording_id_i

        peak_frames.append(peaks_i)
        trace_frames.append(trace_i)
        direction_frames.append(directions_i)
        fit_frames.append(fit_i)
        event_offset += int(result_i.peaks["window_idx"].nunique())

    combined_peaks = pd.concat(peak_frames, ignore_index=True)
    combined_trace = pd.concat(trace_frames, ignore_index=True)
    combined_directions = pd.concat(direction_frames, ignore_index=True)
    per_recording_fit = pd.concat(fit_frames, ignore_index=True)

    combined_bin_summary = summarize_wave_peaks(combined_peaks, min_events_per_bin=WAVE_MIN_EVENTS_PER_BIN)
    if len(combined_bin_summary) < 3:
        raise ValueError("Too few aggregate x-bins survived the minimum-event filter for wave fitting.")
    aggregate_array_width_um = float(per_recording_fit["array_width_um"].median())
    combined_fit_summary, combined_boot_speeds = fit_wave_speed(
        combined_peaks,
        combined_bin_summary,
        x_bin_um=WAVE_X_BIN_UM,
        array_width_um=aggregate_array_width_um,
        event_direction=combined_directions,
        bootstrap_reps=WAVE_BOOTSTRAP_REPS,
        min_events_per_bin=WAVE_MIN_EVENTS_PER_BIN,
        random_seed=WAVE_RANDOM_SEED,
    )
    combined_heatmap = (
        combined_trace.groupby(["time_ms", "origin_x_um"], as_index=False)["rate_hz"]
        .mean()
        .sort_values(["time_ms", "origin_x_um"])
        .pivot(index="time_ms", columns="origin_x_um", values="rate_hz")
        .sort_index()
        .sort_index(axis=1)
    )
    return WaveAnalysisResult(
        event_direction=combined_directions,
        trace=combined_trace,
        peaks=combined_peaks,
        bin_summary=combined_bin_summary,
        fit_summary=combined_fit_summary,
        heatmap=combined_heatmap,
        bootstrap_speeds=combined_boot_speeds,
    ), per_recording_fit


aggregate_wave_results_by_recording = []
aggregate_wave_rows = []
aggregate_prep_cfg = PrepConfig(mode="top", top_start=TOP_START, top_stop=TOP_STOP, verbose=False)

for spec_i in iter_aggregate_recording_specs():
    recording_id_i = spec_i["recording_id"]
    cached_i = None if bool(WAVE_FORCE_RECOMPUTE) else load_wave_result_cache(wave_cache_dir(recording_id_i))
    if cached_i is None:
        ds_i = load_recording_from_spec(spec_i)
        rec_i = ds_i.recordings[0]
        refs_i = ds_i.select_ref_electrodes(aggregate_prep_cfg)[0]
        aligned_i, highres_i, high_epochs_i, burst_epochs_i, high_info_i = build_aligned_for_recording(rec_i, refs_i)
        if len(pd.DataFrame(aligned_i.valid_anchors)) == 0:
            print(f"Skipping {recording_id_i}: no valid aligned burst anchors")
            continue
        result_i = compute_or_load_wave_result(recording_id_i, aligned_i, rec_i.layout, force=bool(WAVE_FORCE_RECOMPUTE))
        n_refs_i = int(len(refs_i))
        n_bursts_i = int(len(burst_epochs_i))
    else:
        print(f"Loaded cached wave result: {wave_cache_dir(recording_id_i)}")
        result_i = cached_i
        n_refs_i = np.nan
        n_bursts_i = int(result_i.fit_summary.iloc[0]["n_events_used"])

    aggregate_wave_results_by_recording.append((recording_id_i, result_i))
    fit_row_i = result_i.fit_summary.iloc[0]
    aggregate_wave_rows.append(
        {
            "recording_id": recording_id_i,
            "dataset": spec_i["dataset"],
            "source_file": spec_i["source_file"],
            "well": int(spec_i["well"]),
            "div": spec_i.get("div", np.nan),
            "n_refs": n_refs_i,
            "n_events_used": int(fit_row_i["n_events_used"]),
            "n_bursts_or_cached_events": n_bursts_i,
            "implied_speed_um_per_ms": float(fit_row_i["implied_speed_um_per_ms"]),
            "bootstrap_speed_mean_um_per_ms": float(fit_row_i["bootstrap_speed_mean_um_per_ms"]) if "bootstrap_speed_mean_um_per_ms" in result_i.fit_summary.columns else float(np.mean(result_i.bootstrap_speeds)),
            "bootstrap_speed_median_um_per_ms": float(fit_row_i["bootstrap_speed_median_um_per_ms"]),
            "bootstrap_speed_ci_low_um_per_ms": float(fit_row_i["bootstrap_speed_ci_low_um_per_ms"]),
            "bootstrap_speed_ci_high_um_per_ms": float(fit_row_i["bootstrap_speed_ci_high_um_per_ms"]),
            "n_events_left_to_right": int(fit_row_i["n_events_left_to_right"]),
            "n_events_right_to_left": int(fit_row_i["n_events_right_to_left"]),
        }
    )

if not aggregate_wave_results_by_recording:
    raise ValueError("No aggregate wave results were available. Check AGGREGATE_WELLS and AGGREGATE_DIVS.")

aggregate_recording_id = f"aggregate_{DATASET}_wells{'-'.join(map(str, AGGREGATE_WELLS))}"
if DATASET == "stim_removal_null":
    aggregate_recording_id += f"_DIVs{'-'.join(map(str, AGGREGATE_DIVS))}"
aggregate_cache_dir = wave_cache_dir(aggregate_recording_id)
aggregate_wave_result = None if bool(AGGREGATE_WAVE_FORCE_RECOMPUTE) else load_wave_result_cache(aggregate_cache_dir)
aggregate_wave_per_recording_fit = None
if aggregate_wave_result is None:
    aggregate_wave_result, aggregate_wave_per_recording_fit = aggregate_wave_results(aggregate_wave_results_by_recording)
    save_wave_result_cache(aggregate_wave_result, aggregate_cache_dir)
    print(f"Saved aggregate wave result cache: {aggregate_cache_dir}")
else:
    print(f"Loaded cached aggregate wave result: {aggregate_cache_dir}")

aggregate_wave_summary = pd.DataFrame(aggregate_wave_rows)

plot_wave_result_figure(aggregate_wave_result, title_prefix=f"Aggregate {DATASET} wells | ", show_tables=False)
display(aggregate_wave_result.fit_summary.round(3))
display(aggregate_wave_summary.round(3))
display(aggregate_wave_result.bin_summary.round(3))


## Transfer Entropy
Transfer-entropy analyses are kept out of this burst notebook. Use `Notebooks/stimRemovalNull_transferEntropy.ipynb` for TE-specific configuration, execution, and plots.


## Example Burst-Anchor-Centered IFR Time Series
Inspect individual event windows centered on the max high-resolution population IFR anchor. The left column shows the same high-resolution population trace used for anchor selection.


## Example Event Hex-Tiled GIFs
Use the same event selection as the example burst-peak-centered IFR section, but render each event as its own hex-tiled propagation GIF. The frame timing, hex-grid size, and log-scale rendering match the later average-array GIF section. GIF saving is optional; when disabled, preview files are written to a temporary directory for display only.


In [ ]:
example_hex_dir = (repo_root / EXAMPLE_EVENT_HEX_DIR) if EXAMPLE_EVENT_HEX_SAVE else (Path('/tmp') / 'ephax_example_event_hex')
example_hex_dir.mkdir(parents=True, exist_ok=True)

example_frame_step = max(1, int(round(GIF_FRAME_STEP_MS / HIGHRES_BIN_MS)))
example_frame_indices = np.arange(0, len(relative_time_ms), example_frame_step, dtype=int)

example_full_layout_df = pd.DataFrame(rec.layout).drop_duplicates("electrode").copy()
example_layout_df = example_full_layout_df[example_full_layout_df["electrode"].isin(selected_electrodes)].copy()
example_hex_x = example_layout_df["x"].to_numpy(dtype=float)
example_hex_y = example_layout_df["y"].to_numpy(dtype=float)
example_hex_electrodes = example_layout_df["electrode"].to_numpy(dtype=int)
example_hex_idx = np.array([np.flatnonzero(selected_electrodes == int(el))[0] for el in example_hex_electrodes], dtype=int)
example_xlim = (ARRAY_X_MIN_UM, ARRAY_X_MAX_UM)
example_ylim = (ARRAY_Y_MIN_UM, ARRAY_Y_MAX_UM)
example_box_aspect = (example_ylim[1] - example_ylim[0]) / max(example_xlim[1] - example_xlim[0], 1e-9)
example_hex_extent = (*example_xlim, *example_ylim)

example_positive = aligned_tensor[aligned_tensor > 0]
if example_positive.size == 0:
    raise ValueError("No positive values were available for event-wise hex GIFs.")
example_vmin = max(1e-2, float(np.quantile(example_positive, 0.05)))
example_vmax = max(example_vmin * 10.0, float(np.quantile(example_positive, 0.995)))

example_event_gif_paths = []
for row in example_events.itertuples(index=False):
    window_idx = int(row.window_idx)
    if window_idx < 0 or window_idx >= len(aligned_tensor):
        continue
    event_tensor = aligned_tensor[window_idx]
    band_label = str(row.selection_band)
    gif_name = f"well{WELL}_DIV{DIV}_event{int(row.coarse_event_idx)}_{band_label}.gif"
    gif_path = example_hex_dir / gif_name

    event_spike_tensor = aligned_spike_tensor[window_idx]

    with imageio.get_writer(gif_path, mode="I", duration=0.14) as writer:
        for time_index in example_frame_indices:
            values = event_tensor[example_hex_idx, int(time_index)]
            spike_mask = event_spike_tensor[example_hex_idx, int(time_index)]
            fig, ax = plt.subplots(figsize=(7.4, 4.6), constrained_layout=True)
            hb = ax.hexbin(
                example_hex_x,
                example_hex_y,
                C=np.clip(values, example_vmin, example_vmax),
                reduce_C_function=np.mean,
                gridsize=HEX_GRID_SIZE,
                cmap="magma",
                mincnt=1,
                linewidths=0.35,
                edgecolors="black",
                norm=LogNorm(vmin=example_vmin, vmax=example_vmax),
                extent=example_hex_extent,
            )
            if np.any(spike_mask):
                spike_hb = ax.hexbin(
                    example_hex_x[spike_mask],
                    example_hex_y[spike_mask],
                    gridsize=HEX_GRID_SIZE,
                    mincnt=1,
                    linewidths=0.35,
                    edgecolors="white",
                    facecolors="none",
                    extent=example_hex_extent,
                )
                spike_hb.set_facecolor("none")
                spike_hb.set_edgecolor("white")
                spike_hb.set_linewidth(0.5)
            ax.set_facecolor("black")
            ax.set_xlim(*example_xlim)
            ax.set_ylim(*example_ylim)
            ax.set_xlabel("x (um)")
            ax.set_ylabel("y (um)")
            ax.set_aspect("equal", adjustable="box")
            ax.set_box_aspect(example_box_aspect)
            ax.set_title(
                f"{band_label.title()} example | event {int(row.coarse_event_idx)} | t = {relative_time_ms[time_index]:.0f} ms"
            )
            fig.colorbar(hb, ax=ax, label="Instantaneous firing rate (Hz)")
            buf = BytesIO()
            fig.savefig(buf, format="png", dpi=140, bbox_inches="tight")
            buf.seek(0)
            writer.append_data(imageio.imread(buf))
            buf.close()
            plt.close(fig)

    example_event_gif_paths.append((band_label, int(row.coarse_event_idx), int(row.gamma_peak_rank), gif_path))

for band_label, coarse_event_idx, gamma_peak_rank, gif_path in example_event_gif_paths:
    print(f"{band_label.title()} example | event {coarse_event_idx} | gamma {gamma_peak_rank}: {gif_path}")
    display(Image(filename=str(gif_path)))


## Gamma-Centered Propagation Summary

Average the high-resolution, per-electrode spike-density traces across all valid nested anchors and sort electrodes by how strongly they fire around the burst peak. This pushes the most active, most rhythmically re-firing electrodes to the top of the view.


In [ ]:
mean_aligned_electrode_rate = aligned_tensor.mean(axis=0)
peak_window_mask = (relative_time_ms >= -SORT_PEAK_PRE_MS) & (relative_time_ms <= SORT_PEAK_POST_MS)
rebound_window_mask = (relative_time_ms >= 15.0) & (relative_time_ms <= 25.0)

rate_summary = pd.DataFrame(
    {
        "electrode": selected_electrodes.astype(int),
        "window_mean_hz": mean_aligned_electrode_rate.mean(axis=1),
        "peak_window_mean_hz": mean_aligned_electrode_rate[:, peak_window_mask].mean(axis=1),
        "rebound_window_mean_hz": mean_aligned_electrode_rate[:, rebound_window_mask].mean(axis=1),
        "anchor_rate_hz": mean_aligned_electrode_rate[:, relative_time_ms == 0.0].reshape(-1),
        "max_rate_hz": mean_aligned_electrode_rate.max(axis=1),
    }
).sort_values(
    ["peak_window_mean_hz", "rebound_window_mean_hz", "max_rate_hz", "electrode"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

ordered_electrodes = rate_summary["electrode"].to_numpy(dtype=int)
ordered_idx = np.array([np.flatnonzero(selected_electrodes == int(el))[0] for el in ordered_electrodes], dtype=int)
ordered_aligned_rate = mean_aligned_electrode_rate[ordered_idx]
top_active_count = min(60, max(10, ordered_aligned_rate.shape[0] // 10))
top_active_trace = ordered_aligned_rate[:top_active_count].mean(axis=0)

positive_aligned = ordered_aligned_rate[ordered_aligned_rate > 0]
aligned_vmin = max(1e-2, float(np.quantile(positive_aligned, 0.05)))
aligned_vmax = max(aligned_vmin * 10.0, float(np.quantile(positive_aligned, 0.995)))

fig = plt.figure(figsize=(13, 7), constrained_layout=True)
gs = fig.add_gridspec(2, 2, height_ratios=[3.0, 1.2], width_ratios=[40.0, 1.6])
ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[1, 0], sharex=ax0)
cax = fig.add_subplot(gs[0, 1])
fig.add_subplot(gs[1, 1]).axis("off")

im = ax0.imshow(
    np.clip(ordered_aligned_rate, aligned_vmin, aligned_vmax),
    aspect="auto",
    origin="lower",
    extent=[relative_time_ms[0], relative_time_ms[-1], 0.5, ordered_aligned_rate.shape[0] + 0.5],
    cmap="viridis",
    norm=LogNorm(vmin=aligned_vmin, vmax=aligned_vmax),
)
ax0.axvline(0.0, color="crimson", ls="--", lw=1.0)
ax0.set_ylabel("Electrodes sorted by mean IFR near the gamma peak")
ax0.set_title("Average per-electrode propagation profile around the nested gamma anchor")

cbar = fig.colorbar(im, cax=cax)
cbar.set_label("Spike-density rate (Hz, log scale)")
cbar.locator = LogLocator(base=10)
cbar.formatter = LogFormatterMathtext(base=10)
cbar.update_ticks()

ax1.plot(relative_time_ms, population_windows.mean(axis=0), color="0.45", lw=1.8, label="All selected electrodes")
ax1.plot(relative_time_ms, top_active_trace, color="black", lw=2.4, label=f"Top {top_active_count} peak-active electrodes")
ax1.axvline(0.0, color="crimson", ls="--", lw=1.0)
ax1.set_xlabel("Time relative to gamma anchor (ms)")
ax1.set_ylabel("Population rate (Hz)")
ax1.set_title("Mean gamma-centered population trace")
ax1.legend(loc="upper right")
save_inline_plot(fig, "average_per_electrode_propagation_peak_activity_sorted")

rate_summary.head(20)


## Peak-Time Ordered Propagation View

Sort electrodes by the timepoint of their own maximal mean IFR relative to the refined burst peak anchor. This emphasizes ordering by when each electrode peaks, rather than by how strongly it fires.


In [ ]:
peak_time_idx = np.argmax(mean_aligned_electrode_rate, axis=1)
peak_time_summary = rate_summary.merge(
    pd.DataFrame(
        {
            "electrode": selected_electrodes.astype(int),
            "peak_time_ms": relative_time_ms[peak_time_idx],
            "peak_rate_hz": mean_aligned_electrode_rate[np.arange(mean_aligned_electrode_rate.shape[0]), peak_time_idx],
        }
    ),
    on="electrode",
    how="left",
).sort_values(
    ["peak_time_ms", "peak_rate_hz", "peak_window_mean_hz", "electrode"],
    ascending=[True, False, False, True],
).reset_index(drop=True)

peak_time_electrodes = peak_time_summary["electrode"].to_numpy(dtype=int)
peak_time_order_idx = np.array([np.flatnonzero(selected_electrodes == int(el))[0] for el in peak_time_electrodes], dtype=int)
peak_time_ordered_rate = mean_aligned_electrode_rate[peak_time_order_idx]

early_cutoff = float(peak_time_summary["peak_time_ms"].quantile(0.25))
late_cutoff = float(peak_time_summary["peak_time_ms"].quantile(0.75))
early_trace = peak_time_ordered_rate[peak_time_summary["peak_time_ms"].to_numpy() <= early_cutoff].mean(axis=0)
late_trace = peak_time_ordered_rate[peak_time_summary["peak_time_ms"].to_numpy() >= late_cutoff].mean(axis=0)

peak_time_positive = peak_time_ordered_rate[peak_time_ordered_rate > 0]
peak_time_vmin = max(1e-2, float(np.quantile(peak_time_positive, 0.05)))
peak_time_vmax = max(peak_time_vmin * 10.0, float(np.quantile(peak_time_positive, 0.995)))

fig = plt.figure(figsize=(13, 7), constrained_layout=True)
gs = fig.add_gridspec(2, 2, height_ratios=[3.0, 1.2], width_ratios=[40.0, 1.6])
ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[1, 0], sharex=ax0)
cax = fig.add_subplot(gs[0, 1])
fig.add_subplot(gs[1, 1]).axis("off")

im = ax0.imshow(
    np.clip(peak_time_ordered_rate, peak_time_vmin, peak_time_vmax),
    aspect="auto",
    origin="lower",
    extent=[relative_time_ms[0], relative_time_ms[-1], 0.5, peak_time_ordered_rate.shape[0] + 0.5],
    cmap="viridis",
    norm=LogNorm(vmin=peak_time_vmin, vmax=peak_time_vmax),
)
ax0.axvline(0.0, color="crimson", ls="--", lw=1.0)
ax0.set_ylabel("Electrodes sorted by their own peak-time latency")
ax0.set_title("Average per-electrode propagation profile ordered by electrode peak time")

cbar = fig.colorbar(im, cax=cax)
cbar.set_label("Spike-density rate (Hz, log scale)")
cbar.locator = LogLocator(base=10)
cbar.formatter = LogFormatterMathtext(base=10)
cbar.update_ticks()

ax1.plot(relative_time_ms, population_windows.mean(axis=0), color="0.55", lw=1.7, label="All selected electrodes")
ax1.plot(relative_time_ms, early_trace, color="#1f77b4", lw=2.1, label="Earliest peak-time quartile")
ax1.plot(relative_time_ms, late_trace, color="#d62728", lw=2.1, label="Latest peak-time quartile")
ax1.axvline(0.0, color="crimson", ls="--", lw=1.0)
ax1.set_xlabel("Time relative to gamma anchor (ms)")
ax1.set_ylabel("Population rate (Hz)")
ax1.set_title("Mean gamma-centered population trace by electrode peak-time group")
ax1.legend(loc="upper right")
save_inline_plot(fig, "average_per_electrode_propagation_peak_time_sorted")

peak_time_summary[["electrode", "peak_time_ms", "peak_rate_hz", "peak_window_mean_hz", "rebound_window_mean_hz"]].head(20)


## Ordering Correlation

Compare each electrode's rank under the activity-based ordering and the peak-time ordering. If the two orderings track each other, points should cluster near the diagonal.


In [ ]:
activity_rank = rate_summary[["electrode"]].copy()
activity_rank["peak_activity_rank"] = np.arange(1, len(activity_rank) + 1, dtype=int)

peak_time_rank = peak_time_summary[["electrode", "peak_time_ms", "peak_rate_hz"]].copy()
peak_time_rank["peak_time_rank"] = np.arange(1, len(peak_time_rank) + 1, dtype=int)

ordering_compare = activity_rank.merge(peak_time_rank, on="electrode", how="inner")
rank_corr = float(np.corrcoef(ordering_compare["peak_activity_rank"], ordering_compare["peak_time_rank"])[0, 1])

fig, ax = plt.subplots(figsize=(7.2, 6.2))
scatter = ax.scatter(
    ordering_compare["peak_activity_rank"],
    ordering_compare["peak_time_rank"],
    c=ordering_compare["peak_time_ms"],
    s=24,
    cmap="coolwarm",
    alpha=0.85,
    edgecolors="none",
)
rank_min = 1
rank_max = len(ordering_compare)
ax.plot([rank_min, rank_max], [rank_min, rank_max], color="black", ls="--", lw=1.2)
ax.set_xlim(rank_min, rank_max)
ax.set_ylim(rank_min, rank_max)
ax.set_xlabel("Rank when sorted by mean IFR near gamma peak")
ax.set_ylabel("Rank when sorted by electrode peak time")
ax.set_title(f"Ordering agreement across electrodes | rank corr = {rank_corr:.3f}")
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Electrode peak time relative to gamma anchor (ms)")
ax.grid(True, alpha=0.25)
save_inline_plot(fig, "electrode_ordering_rank_comparison")

ordering_compare.sort_values("peak_activity_rank").head(20)


## X-Ordered Propagation View

Sort electrodes by their physical `x` position on the array to test the left-to-right propagation hypothesis directly.


In [ ]:
layout_df = pd.DataFrame(rec.layout)
layout_df = layout_df[layout_df["electrode"].isin(selected_electrodes)].drop_duplicates("electrode").copy()
layout_df = layout_df.merge(rate_summary, on="electrode", how="left")

x_sorted_layout = layout_df.sort_values(["x", "y", "peak_window_mean_hz", "electrode"], ascending=[True, True, False, True]).reset_index(drop=True)
x_sorted_electrodes = x_sorted_layout["electrode"].to_numpy(dtype=int)
x_sorted_idx = np.array([np.flatnonzero(selected_electrodes == int(el))[0] for el in x_sorted_electrodes], dtype=int)
x_sorted_rate = mean_aligned_electrode_rate[x_sorted_idx]

left_mask = x_sorted_layout["x"] <= x_sorted_layout["x"].quantile(0.25)
right_mask = x_sorted_layout["x"] >= x_sorted_layout["x"].quantile(0.75)
left_trace = x_sorted_rate[left_mask.to_numpy()].mean(axis=0)
right_trace = x_sorted_rate[right_mask.to_numpy()].mean(axis=0)

x_positive = x_sorted_rate[x_sorted_rate > 0]
x_vmin = max(1e-2, float(np.quantile(x_positive, 0.05)))
x_vmax = max(x_vmin * 10.0, float(np.quantile(x_positive, 0.995)))

fig = plt.figure(figsize=(13, 7), constrained_layout=True)
gs = fig.add_gridspec(2, 2, height_ratios=[3.0, 1.2], width_ratios=[40.0, 1.6])
ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[1, 0], sharex=ax0)
cax = fig.add_subplot(gs[0, 1])
fig.add_subplot(gs[1, 1]).axis("off")

im = ax0.imshow(
    np.clip(x_sorted_rate, x_vmin, x_vmax),
    aspect="auto",
    origin="lower",
    extent=[relative_time_ms[0], relative_time_ms[-1], 0.5, x_sorted_rate.shape[0] + 0.5],
    cmap="viridis",
    norm=LogNorm(vmin=x_vmin, vmax=x_vmax),
)
ax0.axvline(0.0, color="crimson", ls="--", lw=1.0)
ax0.set_ylabel("Electrodes sorted by x position (left to right)")
ax0.set_title("Average per-electrode propagation profile ordered by array x position")

cbar = fig.colorbar(im, cax=cax)
cbar.set_label("Spike-density rate (Hz, log scale)")
cbar.locator = LogLocator(base=10)
cbar.formatter = LogFormatterMathtext(base=10)
cbar.update_ticks()

ax1.plot(relative_time_ms, population_windows.mean(axis=0), color="0.55", lw=1.7, label="All selected electrodes")
ax1.plot(relative_time_ms, left_trace, color="#1f77b4", lw=2.1, label="Leftmost x quartile")
ax1.plot(relative_time_ms, right_trace, color="#d62728", lw=2.1, label="Rightmost x quartile")
ax1.axvline(0.0, color="crimson", ls="--", lw=1.0)
ax1.set_xlabel("Time relative to gamma anchor (ms)")
ax1.set_ylabel("Population rate (Hz)")
ax1.set_title("Mean gamma-centered population trace by x-position group")
ax1.legend(loc="upper right")
save_inline_plot(fig, "average_per_electrode_propagation_x_sorted")

x_sorted_layout[["electrode", "x", "y", "peak_window_mean_hz", "rebound_window_mean_hz"]].head(20)


## Example Burst-Anchor-Centered IFR Time Series
Inspect individual event windows centered on the max high-resolution population IFR anchor. The left column shows the same high-resolution population trace used for anchor selection.


In [ ]:
if len(valid_nested_anchors) == 0:
    raise ValueError("No valid burst-peak-centered windows were available for event-level inspection.")

raw_ifr_data, _, _ = calculate_ifr(
    rec.spikes,
    refs,
    rec.start_time,
    rec.end_time,
)

selected_electrodes = np.asarray(refs, dtype=int)
spike_times_all = np.asarray(rec.spikes["time"], dtype=float)
spike_electrodes_all = np.asarray(rec.spikes["electrode"], dtype=int)
time_bin_half_width_s = 0.5 * float(HIGHRES_BIN_MS) / 1000.0
time_edges_highres = np.concatenate(
    ([float(time_centers_highres[0]) - time_bin_half_width_s], time_centers_highres + time_bin_half_width_s)
)

raw_ifr_highres = []
spike_presence_highres = []
event_interval_midpoints = {}
event_interval_ifr = {}
for electrode in selected_electrodes:
    if int(electrode) in raw_ifr_data:
        ifr_times, ifr_values = raw_ifr_data[int(electrode)]
        value_idx = np.searchsorted(ifr_times, time_centers_highres, side="right") - 1
        value_idx = np.clip(value_idx, 0, len(ifr_values) - 1)
        raw_ifr_highres.append(ifr_values[value_idx])
    else:
        raw_ifr_highres.append(np.zeros_like(time_centers_highres, dtype=float))

    electrode_spike_times = np.sort(
        spike_times_all[
            (spike_electrodes_all == int(electrode))
            & (spike_times_all >= rec.start_time)
            & (spike_times_all <= rec.end_time)
        ]
    )
    spike_counts_highres, _ = np.histogram(electrode_spike_times, bins=time_edges_highres)
    spike_presence_highres.append(spike_counts_highres > 0)

    if electrode_spike_times.size < 2:
        event_interval_midpoints[int(electrode)] = np.array([], dtype=float)
        event_interval_ifr[int(electrode)] = np.array([], dtype=float)
    else:
        isi_s = np.diff(electrode_spike_times)
        valid_isi = isi_s > 0
        event_interval_midpoints[int(electrode)] = 0.5 * (
            electrode_spike_times[:-1][valid_isi] + electrode_spike_times[1:][valid_isi]
        )
        event_interval_ifr[int(electrode)] = 1.0 / isi_s[valid_isi]

raw_ifr_highres = np.asarray(raw_ifr_highres, dtype=float)
population_mean_raw_ifr = raw_ifr_highres.mean(axis=0)
population_mean_raw_ifr_smooth = gaussian_filter1d(
    population_mean_raw_ifr,
    sigma=max(1e-6, float(HIGHRES_SMOOTH_SIGMA_MS) / float(HIGHRES_BIN_MS)),
    mode="nearest",
)
spike_presence_highres = np.asarray(spike_presence_highres, dtype=bool)
aligned_spike_tensor = []
for row in valid_nested_anchors.itertuples(index=False):
    anchor_idx = int(np.argmin(np.abs(time_centers_highres - row.anchor_time_s)))
    window_start = anchor_idx - pre_bins
    window_stop = anchor_idx + post_bins + 1
    if window_start < 0 or window_stop > len(time_centers_highres):
        continue
    aligned_spike_tensor.append(spike_presence_highres[:, window_start:window_stop])
aligned_spike_tensor = np.asarray(aligned_spike_tensor, dtype=bool)

all_interval_ifr = raw_ifr_highres[(raw_ifr_highres > 0) & (raw_ifr_highres <= IFR_MAX_HZ)].astype(float)
if all_interval_ifr.size == 0:
    raise ValueError("No positive raw IFR values were found for the selected electrodes.")
global_hist_bins_hz = np.logspace(
    np.log10(max(1e-3, float(all_interval_ifr.min()))),
    np.log10(float(all_interval_ifr.max())),
    60,
)

example_events = select_example_events(valid_nested_anchors, EXAMPLE_GAMMA_EVENT_COUNT)
example_event_count = len(example_events)
if example_event_count == 0:
    raise ValueError("No example events were available for event-level inspection.")

fig, axes = plt.subplots(
    example_event_count,
    3,
    figsize=(19, max(3.2 * example_event_count, 4.0)),
    constrained_layout=True,
    squeeze=False,
)

for row_idx, row in enumerate(example_events.itertuples(index=False)):
    anchor_idx = int(np.argmin(np.abs(time_centers_highres - row.anchor_time_s)))
    window_start = anchor_idx - pre_bins
    window_stop = anchor_idx + post_bins + 1
    if window_start < 0 or window_stop > len(time_centers_highres):
        continue

    electrode_window = raw_ifr_highres[:, window_start:window_stop]
    pop_trace = highres.population_rate_hz[window_start:window_stop]
    absolute_window_start = float(time_centers_highres[window_start])
    absolute_window_stop = float(time_centers_highres[window_stop - 1])
    event_interval_ifr_samples = []
    for electrode in selected_electrodes:
        mids = event_interval_midpoints[int(electrode)]
        vals = event_interval_ifr[int(electrode)]
        in_window = (mids >= absolute_window_start) & (mids <= absolute_window_stop)
        if np.any(in_window):
            event_interval_ifr_samples.append(vals[in_window])
    event_interval_ifr_samples = (
        np.concatenate(event_interval_ifr_samples).astype(float)
        if event_interval_ifr_samples
        else np.array([], dtype=float)
    )

    ax_line = axes[row_idx, 0]
    ax_heat = axes[row_idx, 1]
    ax_hist = axes[row_idx, 2]

    ax_line.plot(relative_time_ms, pop_trace, color="black", lw=1.6)
    ax_line.axvline(0.0, color="crimson", ls="--", lw=1.1)
    ax_line.set_ylabel("High-res population IFR (Hz)")
    ax_line.set_title(
        f"{str(row.selection_band).title()} example | event {int(row.coarse_event_idx)} | anchor {int(row.gamma_peak_rank)}"
    )
    ax_line.set_xlim(float(relative_time_ms[0]), float(relative_time_ms[-1]))

    positive_vals = electrode_window[electrode_window > 0]
    if positive_vals.size == 0:
        heat_vmin = 1e-3
        heat_vmax = 1.0
    else:
        heat_vmin = max(1e-3, float(np.quantile(positive_vals, 0.02)))
        heat_vmax = max(heat_vmin * 10.0, float(np.quantile(positive_vals, 0.995)))
    display_window = np.clip(electrode_window, heat_vmin, heat_vmax)

    im = ax_heat.imshow(
        display_window,
        aspect="auto",
        origin="lower",
        extent=[float(relative_time_ms[0]), float(relative_time_ms[-1]), 0.5, electrode_window.shape[0] + 0.5],
        cmap="viridis",
        norm=LogNorm(vmin=heat_vmin, vmax=heat_vmax),
    )
    ax_heat.axvline(0.0, color="crimson", ls="--", lw=1.0)
    ax_heat.set_title("Per-electrode raw IFR")
    ax_heat.set_ylabel("Electrode rank")
    ax_heat.set_yticks([1, electrode_window.shape[0]])
    ax_heat.set_xlim(float(relative_time_ms[0]), float(relative_time_ms[-1]))

    event_interval_ifr_samples = event_interval_ifr_samples[
        np.isfinite(event_interval_ifr_samples)
        & (event_interval_ifr_samples > 0)
        & (event_interval_ifr_samples <= IFR_MAX_HZ)
    ]
    if event_interval_ifr_samples.size > 0:
        ax_hist.hist(
            event_interval_ifr_samples,
            bins=global_hist_bins_hz,
            color="0.30",
            edgecolor="white",
            linewidth=0.25,
        )
    ax_hist.set_title(f"Event raw IFR histogram (n={int(event_interval_ifr_samples.size)})")
    ax_hist.set_ylabel("Frequency")
    ax_hist.grid(True, axis="x", alpha=0.18)
    ax_hist.set_xscale("log")

    if row_idx == example_event_count - 1:
        ax_line.set_xlabel("Time relative to max-IFR burst anchor (ms)")
        ax_heat.set_xlabel("Time relative to max-IFR burst anchor (ms)")
        ax_hist.set_xlabel("Instantaneous firing rate (Hz)")

    cbar = fig.colorbar(im, ax=ax_heat, fraction=0.046, pad=0.02)
    cbar.set_label("IFR (Hz, log scale)")
    cbar.locator = LogLocator(base=10)
    cbar.formatter = LogFormatterMathtext(base=10)
    cbar.update_ticks()

save_inline_plot(fig, "example_event_raw_ifr_inspection")

example_events[[
    "coarse_event_idx",
    "gamma_peak_rank",
    "anchor_time_s",
    "anchor_height_hz",
    "gamma_anchor_delay_ms",
    "subpeak_count",
    "selection_band",
]].round(3)


## Next Step

The notebook now uses nested anchoring, ranks electrodes by peak-window IFR, shows an x-ordered propagation view, and renders a coarse hex-binned burst-peak-centered GIF. The next useful refinement is to compare this average-array GIF against event-specific GIFs from only the strongest bursts, so that propagation structure is not averaged away.
